# Contract Gate

Run contract and artifact gate checks.

Steps:
- Run contract diff and artifact gate scripts.
- Review output reports.
- Summarize gate status.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
summary = {
    'contract_diff_exit': None,
    'artifact_gate_exit': None,
    'ci_gate_exit': None,
    'reports': {},
}


def run_optional(cmd: list[str]) -> int:
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=str(REPO_ROOT), env=env)
    print('Return code:', result.returncode)
    return result.returncode


In [ ]:
# Run contract diff and artifact gate.
contract_script = REPO_ROOT / 'scripts' / 'contract_diff.py'
openapi_path = REPO_ROOT / 'reports' / 'openapi.json'
if contract_script.exists() and openapi_path.exists() and openapi_path.stat().st_size > 0:
    summary['contract_diff_exit'] = run_optional([PY, 'scripts/contract_diff.py'])
else:
    print('Skipping contract diff; missing or empty reports/openapi.json')

artifact_gate = REPO_ROOT / 'scripts' / 'artifact_gate.py'
if artifact_gate.exists():
    summary['artifact_gate_exit'] = run_optional([PY, 'scripts/artifact_gate.py'])
else:
    print('Missing:', artifact_gate)

ci_gate = REPO_ROOT / 'scripts' / 'ci_contract_gate.sh'
if ci_gate.exists():
    summary['ci_gate_exit'] = run_optional(['bash', 'scripts/ci_contract_gate.sh'])
else:
    print('Missing:', ci_gate)


In [ ]:
# Review output reports.
reports = {
    'CONTRACT_DIFF.md': REPO_ROOT / 'reports' / 'CONTRACT_DIFF.md',
    'ARTIFACT_GATE.md': REPO_ROOT / 'reports' / 'ARTIFACT_GATE.md',
    'ARTIFACT_GATE.json': REPO_ROOT / 'reports' / 'ARTIFACT_GATE.json',
}

for name, path in reports.items():
    if path.exists():
        summary['reports'][name] = str(path.relative_to(REPO_ROOT))
        print('')
        print(name)
        print(path.read_text(encoding='utf-8', errors='ignore')[:1200])
    else:
        print('Missing:', path)


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'evaluation_contract_gate_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
